# Xử lý dữ liệu Thần kinh — Phục dựng điểm GCS (Glasgow Coma Scale)

**Mục tiêu:** Tái tạo điểm GCS tổng hợp từ 3 thành phần APACHE (`eyes`, `motor`, `verbal`) trong bảng `apacheApsVar.csv`.

**Input:**
- `raw/eicu/apacheApsVar.csv` — dữ liệu APACHE gốc
- `processed/patient_clean.csv` — danh sách bệnh nhân hợp lệ (cohort)

**Output:** `processed/gcs_clean.csv`

**GCS (Glasgow Coma Scale):**
- `eyes` (mở mắt): 1–4
- `motor` (vận động): 1–6
- `verbal` (lời nói): 1–5
- Tổng: **3–15** (3 = hôn mê sâu, 15 = tỉnh táo hoàn toàn)

In [3]:
# ============================================================
# BƯỚC 1: KHỞI TẠO VÀ NẠP DỮ LIỆU
# ============================================================
# Nạp 2 file:
#   - apacheApsVar.csv : chứa 3 thành phần GCS (eyes, motor, verbal)
#   - patient_clean.csv: danh sách patientunitstayid hợp lệ (cohort)
#
# Mục đích: chỉ giữ những bệnh nhân đã qua lọc cohort ở bước patient.

import pandas as pd
import numpy as np

# Đường dẫn (tính từ data_processing/)
APACHE_PATH  = "../data/raw/eicu/apacheApsVar.csv"
PATIENT_PATH = "../data/processed/patient_clean.csv"
OUTPUT_PATH  = "../data/processed/gcs_clean.csv"

# Đọc dữ liệu APACHE gốc
df_apache = pd.read_csv(APACHE_PATH)
print("Kích thước apacheApsVar gốc:", df_apache.shape)
print("\n5 dòng đầu tiên:")
display(df_apache.head())

# Đọc danh sách bệnh nhân hợp lệ
df_patient = pd.read_csv(PATIENT_PATH)
valid_ids = set(df_patient["patientunitstayid"])
print(f"\nSố bệnh nhân hợp lệ trong cohort: {len(valid_ids):,}")

Kích thước apacheApsVar gốc: (2205, 26)

5 dòng đầu tiên:


,apacheapsvarid,patientunitstayid,intubated,vent,dialysis,eyes,motor,verbal,meds,urine,...,ph,hematocrit,creatinine,albumin,pao2,pco2,bun,glucose,bilirubin,fio2
0,92788,141765,0,0,0,4,6,5,0,-1.0,...,-1.0,37.8,1.04,-1.0,-1.0,-1.0,28.0,61,-1.0,-1
1,8893,143870,0,0,0,4,6,5,0,-1.0,...,-1.0,34.1,1.14,-1.0,-1.0,-1.0,14.0,140,-1.0,-1
2,79585,144815,0,0,0,4,6,5,0,-1.0,...,-1.0,36.6,0.63,3.6,-1.0,-1.0,6.0,82,0.5,-1
3,203242,145427,0,0,0,4,6,5,0,-1.0,...,-1.0,40.4,1.05,-1.0,-1.0,-1.0,14.0,118,-1.0,-1
4,154681,147307,0,0,0,4,6,5,0,-1.0,...,-1.0,-1.0,-1.00,-1.0,-1.0,-1.0,-1.0,-1,-1.0,-1



Số bệnh nhân hợp lệ trong cohort: 2,326


In [4]:
# ============================================================
# BƯỚC 2: CHỌN CỘT THÀNH PHẦN GCS
# ============================================================
# Chỉ giữ 4 cột cần thiết:
#   - patientunitstayid : khóa chính để JOIN
#   - eyes              : điểm mở mắt (1–4)
#   - motor             : điểm vận động (1–6)
#   - verbal            : điểm lời nói (1–5)
#
# Các cột khác trong apacheApsVar (như bilirubin, creatinine...)
# không dùng cho GCS -> loại bỏ để tiết kiệm bộ nhớ.

GCS_COLS = ["patientunitstayid", "eyes", "motor", "verbal"]
df_gcs = df_apache[GCS_COLS].copy()

print("Kích thước sau khi chọn cột:", df_gcs.shape)
print("\nThông tin kiểu dữ liệu:")
df_gcs.info()

print("\nThống kê mô tả TRƯỚC khi xử lý:")
print(df_gcs[["eyes", "motor", "verbal"]].describe())

Kích thước sau khi chọn cột: (2205, 4)

Thông tin kiểu dữ liệu:
<class 'pandas.DataFrame'>
RangeIndex: 2205 entries, 0 to 2204
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   patientunitstayid  2205 non-null   int64
 1   eyes               2205 non-null   int64
 2   motor              2205 non-null   int64
 3   verbal             2205 non-null   int64
dtypes: int64(4)
memory usage: 69.0 KB

Thống kê mô tả TRƯỚC khi xử lý:
              eyes        motor       verbal
count  2205.000000  2205.000000  2205.000000
mean      3.388662     5.282993     3.980499
std       1.269479     1.769295     1.749016
min      -1.000000    -1.000000    -1.000000
25%       3.000000     6.000000     4.000000
50%       4.000000     6.000000     5.000000
75%       4.000000     6.000000     5.000000
max       4.000000     6.000000     5.000000


In [5]:
# ============================================================
# BƯỚC 3: XỬ LÝ DỮ LIỆU THIẾU / BẤT THƯỜNG (-1 -> NaN)
# ============================================================
# Trong eICU, giá trị âm (< 0, thường là -1) là mã sentinel
# đại diện cho "không đánh giá được" hoặc "dữ liệu bị thiếu".
# Đây KHÔNG phải điểm số lâm sàng hợp lệ.
#
# Nếu để nguyên -1, khi cộng GCS sẽ ra kết quả vô lý (ví dụ -3).
# => Phải chuyển tất cả giá trị < 0 thành NaN.

# Đếm số giá trị âm TRƯỚC khi xử lý (để báo cáo)
for col in ["eyes", "motor", "verbal"]:
    n_neg = (df_gcs[col] < 0).sum()
    print(f"Số giá trị âm ở {col:7s}: {n_neg:,} dòng")

# Chuyển tất cả giá trị < 0 thành NaN
df_gcs.loc[df_gcs["eyes"]   < 0, "eyes"]   = np.nan
df_gcs.loc[df_gcs["motor"]  < 0, "motor"]  = np.nan
df_gcs.loc[df_gcs["verbal"] < 0, "verbal"] = np.nan

print("\nThống kê mô tả SAU khi xử lý (-1 -> NaN):")
print(df_gcs[["eyes", "motor", "verbal"]].describe())

# Kiểm tra phạm vi hợp lệ
print("\nKiểm tra phạm vi hợp lệ:")
print(f"  eyes   hợp lệ [1, 4]: "
      f"{df_gcs['eyes'].dropna().between(1, 4).all()}")
print(f"  motor  hợp lệ [1, 6]: "
      f"{df_gcs['motor'].dropna().between(1, 6).all()}")
print(f"  verbal hợp lệ [1, 5]: "
      f"{df_gcs['verbal'].dropna().between(1, 5).all()}")

Số giá trị âm ở eyes   : 100 dòng
Số giá trị âm ở motor  : 100 dòng
Số giá trị âm ở verbal : 100 dòng

Thống kê mô tả SAU khi xử lý (-1 -> NaN):
              eyes        motor       verbal
count  2105.000000  2105.000000  2105.000000
mean      3.597150     5.581473     4.217102
std       0.853973     1.146220     1.403381
min       1.000000     1.000000     1.000000
25%       4.000000     6.000000     4.000000
50%       4.000000     6.000000     5.000000
75%       4.000000     6.000000     5.000000
max       4.000000     6.000000     5.000000

Kiểm tra phạm vi hợp lệ:
  eyes   hợp lệ [1, 4]: True
  motor  hợp lệ [1, 6]: True
  verbal hợp lệ [1, 5]: True


In [6]:
# ============================================================
# BƯỚC 4: PHỤC DỰNG ĐIỂM GCS (GLASGOW COMA SCALE)
# ============================================================
# Công thức: gcs = eyes + motor + verbal
# Phạm vi hợp lệ: 3 (hôn mê sâu) đến 15 (tỉnh táo hoàn toàn)
#
# Lưu ý: pandas tự động cho kết quả NaN nếu BẤT KỲ thành phần
# nào là NaN (vì NaN + số = NaN). Điều này là mong muốn:
#   - Nếu thiếu 1 thành phần -> không thể tính GCS chính xác
#   - Giữ NaN để XGBoost tự xử lý (như đã bàn)

df_gcs["gcs"] = df_gcs["eyes"] + df_gcs["motor"] + df_gcs["verbal"]

print("Thống kê cột GCS sau phục dựng:")
print(df_gcs["gcs"].describe())

print(f"\nSố dòng GCS bị NaN: {df_gcs['gcs'].isna().sum():,} "
      f"({df_gcs['gcs'].isna().mean()*100:.2f}%)")

# Kiểm tra phạm vi hợp lệ
gcs_valid = df_gcs["gcs"].dropna()
print(f"\nMin GCS: {gcs_valid.min():.0f}  (phải >= 3)")
print(f"Max GCS: {gcs_valid.max():.0f}  (phải <= 15)")

n_out_of_range = ((gcs_valid < 3) | (gcs_valid > 15)).sum()
print(f"{'✅' if n_out_of_range == 0 else '⚠️'} "
      f"Số GCS ngoài [3, 15]: {n_out_of_range}")

# Phân bố GCS
print("\nPhân bố điểm GCS (value_counts):")
print(df_gcs["gcs"].value_counts().sort_index().to_string())

Thống kê cột GCS sau phục dựng:
count    2105.000000
mean       13.395724
std         3.116661
min         3.000000
25%        14.000000
50%        15.000000
75%        15.000000
max        15.000000
Name: gcs, dtype: float64

Số dòng GCS bị NaN: 100 (4.54%)

Min GCS: 3  (phải >= 3)
Max GCS: 15  (phải <= 15)
✅ Số GCS ngoài [3, 15]: 0

Phân bố điểm GCS (value_counts):
gcs
3.0       90
4.0        3
5.0        4
6.0       37
7.0       39
8.0       44
9.0       40
10.0      39
11.0      55
12.0      59
13.0     109
14.0     221
15.0    1365


In [8]:
# ============================================================
# BƯỚC 5: KHỚP COHORT BỆNH NHÂN VÀ XUẤT FILE
# ============================================================
# Chỉ giữ những patientunitstayid có trong patient_clean.csv
# (đã qua lọc cohort: người lớn, nằm ICU >= 4h, có nhãn).

n_before = len(df_gcs)

# Lọc theo cohort
df_gcs = df_gcs[df_gcs["patientunitstayid"].isin(valid_ids)].copy()

n_after = len(df_gcs)
print(f"Số dòng apache gốc         : {n_before:,}")
print(f"Sau khi khớp cohort        : {n_after:,}")
print(f"Đã loại bỏ                 : {n_before - n_after:,} dòng")

# ---------- Kiểm tra tính duy nhất ----------
n_dup = df_gcs["patientunitstayid"].duplicated().sum()
print(f"\nSố patientunitstayid trùng lặp: {n_dup}")

if n_dup > 0:
    print("⚠️ Có trùng lặp — cần xử lý (giữ dòng đầu tiên):")
    df_gcs = df_gcs.drop_duplicates(subset="patientunitstayid", keep="first")
    print(f"   Sau khi drop_duplicates: {len(df_gcs):,} dòng")

assert df_gcs["patientunitstayid"].duplicated().sum() == 0, \
    "Vẫn còn trùng lặp!"

# ---------- Chỉ giữ 2 cột final ----------
df_final = df_gcs[["patientunitstayid", "gcs"]].copy()

print("\nKích thước cuối:", df_final.shape)
print("\n5 dòng đầu tiên:")
print(df_final.head())

# ---------- Thống kê GCS cuối ----------
print("\nThống kê GCS cuối cùng:")
print(df_final["gcs"].describe())

n_gcs_missing = df_final["gcs"].isna().sum()
print(f"\nSố bản ghi thiếu GCS: {n_gcs_missing:,} "
      f"({n_gcs_missing/len(df_final)*100:.2f}%)")
print("→ XGBoost sẽ tự xử lý NaN, không cần điền hoặc bỏ.")

# ---------- Lưu file ----------
import os
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df_final.to_csv(OUTPUT_PATH, index=False)

print(f"\n✅ Đã lưu file: {OUTPUT_PATH}")
print(f"   Kích thước cuối: {df_final.shape}")

Số dòng apache gốc         : 2,074
Sau khi khớp cohort        : 2,074
Đã loại bỏ                 : 0 dòng

Số patientunitstayid trùng lặp: 0

Kích thước cuối: (2074, 2)

5 dòng đầu tiên:
   patientunitstayid   gcs
0             141765  15.0
1             143870  15.0
2             144815  15.0
3             145427  15.0
4             147307  15.0

Thống kê GCS cuối cùng:
count    2023.000000
mean       13.438952
std         3.062021
min         3.000000
25%        14.000000
50%        15.000000
75%        15.000000
max        15.000000
Name: gcs, dtype: float64

Số bản ghi thiếu GCS: 51 (2.46%)
→ XGBoost sẽ tự xử lý NaN, không cần điền hoặc bỏ.

✅ Đã lưu file: ../data/processed/gcs_clean.csv
   Kích thước cuối: (2074, 2)
